Вы работаете аналитиком в приложении для онлайн-знакомств. Механика приложения следующая: пользователи видят в приложении анкеты друг друга и ставят лайки или дизлайки. Если пользователи поставили друг другу лайк – это мэтч. У пользователей появляется возможность познакомиться.


Команда приложения разработала новый алгоритм для поиска наиболее подходящих анкет. Для проверки работы алгоритма провели A/B-тест. Все пользователи были разделены на две группы. Пользователи в группе с номером 0 пользовались приложением со старым алгоритмом. Пользователи в группе №1 пользовались приложением с новым алгоритмом для поиска анкет.


Ваша задача: оценить, действительно ли новый алгоритм улучшил качество сервиса. Для этого нужно выбрать метрики, которые отвечают за качество сервиса, и статистически сравнить эти метрики в двух группах.


В данных находится выгрузка логов взаимодействия пользователей друг с другом. Для каждой пары пользователей указаны их группа в A/B-тесте и факт мэтча.


Результат вашей работы – аналитическое заключение с ответом на вопрос, стоит ли включать новую систему поиска анкет на всех пользователей.


In [159]:
import pandas as pd
from operator import attrgetter
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib import colors as mcolors
import numpy as np

In [172]:
data = pd.read_csv('dating_data.csv')

In [173]:
data.head()

,user_id_1,user_id_2,group,is_match
0,79,91,1,1
1,716,353,1,1
2,423,677,0,0
3,658,165,1,1
4,969,155,0,1


Чему равна средняя доля мэтчей на пользователя в 0 группе? Ответ нужно предоставить до 4-х знаков после запятой без округлений.

In [174]:
# Группируем по user_id_1 и считаем долю мэтчей
match_rates_0 = data[data.group == 0].groupby('user_id_1').apply(
    lambda x: round(x['is_match'].sum() / len(x), 2)
).reset_index(name='match_rate_rounded')

average_match_rate_0 = match_rates_0['match_rate_rounded'].mean()


C:\Users\user\AppData\Local\Temp\ipykernel_19072\3276173568.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  match_rates_0 = data[data.group == 0].groupby('user_id_1').apply(


Чему равна средняя доля мэтчей на пользователя в 1 группе? Ответ нужно предоставить до 4-х знаков после запятой без округлений.

In [175]:
# Группируем по user_id_1 и считаем долю мэтчей
match_rates_1 = data[data.group == 1].groupby('user_id_1').apply(
    lambda x: round(x['is_match'].sum() / len(x), 2)
).reset_index(name='match_rate_rounded')

average_match_rate_1 = match_rates_1['match_rate_rounded'].mean()


C:\Users\user\AppData\Local\Temp\ipykernel_19072\1134493404.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  match_rates_1 = data[data.group == 1].groupby('user_id_1').apply(


|Проверьте, есть ли различия в средней доле мэтчей на пользователя в двух полученных группах. 

In [176]:
import scipy.stats as st

#достаём значения двух групп
control = match_rates_0.match_rate_rounded
test = match_rates_1.match_rate_rounded

#сам тест
st.ttest_ind(control, test, equal_var=False)

TtestResult(statistic=np.float64(-26.481431782585016), pvalue=np.float64(7.890669157070396e-117), df=np.float64(973.9371185920583))

Проверьте, есть ли различия в среднем количестве действий на пользователя в двух полученных группах. Чему равно значение среднего количества действий в 0 группе? Ответ нужно предоставить до 4-х знаков после запятой без округлений.

In [189]:
action_avg_0 = data[data.group == 0].groupby('user_id_1', as_index=False).size()
action_avg_0.mean()

user_id_1    497.371257
size           9.564870
dtype: float64

Чему равно значение среднего количества действий в 1 группе? Ответ нужно предоставить до 4-х знаков после запятой без округлений.

In [190]:
action_avg_1 = data[data.group == 1].groupby('user_id_1', as_index=False).size()
action_avg_1.mean()

user_id_1    503.641283
size          19.482966
dtype: float64

In [ ]:
#достаём значения двух групп
control = action_avg_0['size']
test = action_avg_1['size']

#сам тест
st.ttest_ind(control, test, equal_var=False)


TtestResult(statistic=np.float64(-51.85383774946492), pvalue=np.float64(1.8942877064043142e-285), df=np.float64(998.0))

Для проверки гипотезы о равенстве среднего количества действий я использую 
t-критерий Стьюдента
, поскольку переменные обе — 
количественные
. Нулевая гипотеза 
отклоняется
, поскольку p-value 
меньше
 0.05.

In [199]:
st.ttest_ind(control, test, equal_var=True)

TtestResult(statistic=np.float64(-51.85383774946492), pvalue=np.float64(1.8942877064043142e-285), df=np.float64(998.0))